# 09-практикалық жұмыс — Сызықтық регрессияға кіріспе
**Google Colab | 18 қадам**  
Пән: Желілік қауіптерді анықтауға арналған машиналық оқыту  
**Аты-жөні:** Куанткан Шапагат  
**Тобы:** 24-ПО-103-3К-5



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

print('Сызықтық регрессия жұмысына дайын!')

In [ ]:
rng = np.random.default_rng(42)
n_rows = 60

duration_sec = rng.integers(2, 61, n_rows)
noise = rng.normal(0, 12, n_rows)

# Жалпы бағыт: уақыт артқан сайын KB өседі
bytes_kb = 10 + 2.7 * duration_sec + noise
bytes_kb = np.maximum(bytes_kb, 1).round(1)

df = pd.DataFrame({
    'session_id': [f'R{i:03d}' for i in range(1, n_rows + 1)],
    'duration_sec': duration_sec,
    'bytes_kb': bytes_kb
})

df.to_csv('network_regression_data.csv', index=False)
print('Кесте өлшемі:', df.shape)
display(df.head())

In [ ]:
print('Бос мәндер:')
print(df.isna().sum())
print('\nҚайталанған жолдар:', df.duplicated().sum())
print('\nДерек түрлері:')
print(df.dtypes)
print('\nНегізгі статистика:')
display(df[['duration_sec', 'bytes_kb']].describe().round(2))

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['duration_sec'], df['bytes_kb'], color='royalblue', alpha=0.75, edgecolor='black')
plt.title('Қосылым ұзақтығы және жіберілген дерек көлемі')
plt.xlabel('Қосылым ұзақтығы, секунд')
plt.ylabel('Жіберілген дерек, KB')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('01_initial_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X = df[['duration_sec']].copy()
y = df['bytes_kb'].copy()
print('X shape:', X.shape)
print('y shape:', y.shape)
print('X түрі:', type(X).__name__)
print('y түрі:', type(y).__name__)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

In [ ]:
model = LinearRegression()
print('Модель:', model)
print('Модель жасалды, бірақ әлі үйретілген жоқ')

In [ ]:
model.fit(X_train, y_train)
print('Модель X_train және y_train арқылы үйретілді')

In [ ]:
coefficient = float(model.coef_[0])
intercept = float(model.intercept_)
print('Коэффициент:', round(coefficient, 4))
print('Intercept:', round(intercept, 4))
print(f'Теңдеу: bytes_kb = {coefficient:.2f} × duration_sec + {intercept:.2f}')

In [ ]:
y_pred = model.predict(X_test)
print('Болжам саны:', len(y_pred))
print('Алғашқы бес болжам:', y_pred[:5].round(2))

In [ ]:
predictions = pd.DataFrame({
    'duration_sec': X_test['duration_sec'],
    'actual_bytes_kb': y_test,
    'predicted_bytes_kb': y_pred
})
predictions = predictions.sort_values('duration_sec').reset_index(drop=True)
predictions['predicted_bytes_kb'] = predictions['predicted_bytes_kb'].round(2)
display(predictions.head(10))

In [ ]:
x_line = pd.DataFrame({
    'duration_sec': np.linspace(df['duration_sec'].min(), df['duration_sec'].max(), 100)
})
y_line = model.predict(x_line)

plt.figure(figsize=(10, 6))
plt.scatter(X_train['duration_sec'], y_train, color='royalblue', alpha=0.65, label='Train нүктелері')
plt.scatter(X_test['duration_sec'], y_test, color='darkorange', edgecolor='black', label='Test нүктелері')
plt.plot(x_line['duration_sec'], y_line, color='crimson', linewidth=3, label='Регрессия сызығы')
plt.title('Сызықтық регрессия моделі')
plt.xlabel('Қосылым ұзақтығы, секунд')
plt.ylabel('Жіберілген дерек, KB')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('02_regression_line.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
new_duration = 25
new_data = pd.DataFrame({'duration_sec': [new_duration]})
new_prediction = float(model.predict(new_data)[0])
print('Жаңа ұзақтық:', new_duration, 'секунд')
print('Болжанған көлем:', round(new_prediction, 2), 'KB')

In [ ]:
manual_prediction = coefficient * new_duration + intercept
print('model.predict нәтижесі:', round(new_prediction, 4))
print('Теңдеумен есептеу:', round(manual_prediction, 4))
print('Айырмасы:', round(abs(new_prediction - manual_prediction), 10))

In [ ]:
new_durations = pd.DataFrame({'duration_sec': [5, 10, 20, 30, 45, 60]})
new_durations['predicted_bytes_kb'] = model.predict(new_durations)
new_durations['predicted_bytes_kb'] = new_durations['predicted_bytes_kb'].round(2)
display(new_durations)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['duration_sec'], df['bytes_kb'], color='lightsteelblue', edgecolor='steelblue', label='Бастапқы дерек')
plt.plot(x_line['duration_sec'], y_line, color='crimson', linewidth=3, label='Регрессия сызығы')
plt.scatter(new_durations['duration_sec'], new_durations['predicted_bytes_kb'], color='limegreen', edgecolor='black', s=110, marker='D', label='Жаңа болжамдар')
for _, row in new_durations.iterrows():
    plt.annotate(f"{row['predicted_bytes_kb']:.1f}", (row['duration_sec'], row['predicted_bytes_kb']), xytext=(5, 8), textcoords='offset points')
plt.title('Жаңа мәндерге регрессиялық болжам')
plt.xlabel('Қосылым ұзақтығы, секунд')
plt.ylabel('Жіберілген дерек, KB')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('03_new_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def predict_bytes(model, duration_sec):
    if duration_sec < 0:
        raise ValueError('Ұзақтық теріс болмауы керек')
    data = pd.DataFrame({'duration_sec': [duration_sec]})
    prediction = float(model.predict(data)[0])
    return round(prediction, 2)

for duration in [8, 15, 27, 50]:
    result = predict_bytes(model, duration)
    print(duration, 'секунд →', result, 'KB')

In [ ]:
predictions.to_csv('test_predictions.csv', index=False)
result_files = [
    'network_regression_data.csv',
    '01_initial_scatter.png',
    '02_regression_line.png',
    '03_new_predictions.png',
    'test_predictions.csv'
]
archive_name = 'ML_09_results.zip'
with zipfile.ZipFile(archive_name, 'w') as archive:
    for file_name in result_files:
        archive.write(file_name)
print('Архив дайын:', archive_name)

try:
    from google.colab import files
    files.download(archive_name)
except ImportError:
    print('Colab емес орта: архив ағымдағы бумаға сақталды.')